# 第 1 周末练习 — VERSION 2（改进版）

# 第 1 周 — LLM 解释器：云端 vs 本地 + 评判器（Judge）

## 练习目标（理念）

构建一个小工具，用来对比「云端 OpenAI」与「本地 Ollama」对同一技术问题的解释质量：

1. 把**同一道技术题**分别发给 **云端 OpenAI 模型** 与 **本地 Ollama 模型**
2. 用**流式（streaming）**把两边回答边生成边显示，方便肉眼对比
3. 再用**第三个 GPT 评判器（Judge）**给两边打分（0–10），并选出胜者

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么答」，user 放代码讲解任务 |
| 流式输出 `stream=True` | `stream_answer` + `update_display` 边收边刷新 Markdown |
| OpenAI 云端模型 | `MODEL_CLOUD` / `MODEL_JUDGE` |
| Ollama 本地（OpenAI 兼容） | `base_url=http://localhost:11434/v1`，`MODEL_LOCAL` |
| 结构化输出 | Judge 使用 `response_format={"type": "json_object"}` |

## v2 相对改进点

- `stream_answer` **返回完整回答字符串**（不只是显示）
- 对 API 调用与 Ollama 连接增加错误处理
- 命名统一为英文标识符（可运行逻辑保持一致）
- 对 Judge 的 JSON 回复做校验（字段、分数范围、winner）
- 更完整的文档字符串与类型标注（Type Hints）

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. `.env` 里准备好 `OPENAI_API_KEY`；本机需已启动 Ollama（`ollama serve`）并拉取 `deepseek-r1:8b`
3. 可在「提问」单元格改写 `question`，再分别跑云端流式、本地流式、非流式取全文、Judge 打分


In [1]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 导入标准库 json：解析 Judge 返回的 JSON 裁决结果
import json
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：Markdown 渲染、display / update_display 做流式刷新
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：云端与本地（Ollama OpenAI 兼容端点）都用它
from openai import OpenAI
# 从 typing 导入类型标注：Dict / List / Any / Optional，方便读函数签名
from typing import Dict, List, Any, Optional


In [2]:
# ========== 常量：模型名字集中写在一处，后面只改这里 ==========

# 云端答题模型（OpenAI）：便宜、速度快，适合做对比里的「模型 A」
MODEL_CLOUD = 'gpt-4.1-nano'
# 本地答题模型（Ollama）：需事先 ollama pull；字符串必须和本机已安装的模型名一致
MODEL_LOCAL = 'deepseek-r1:8b'
# 评判器模型（Judge）：通常选更强一点的云端模型，用来打分与选胜者
MODEL_JUDGE = "gpt-4.1-mini"


In [3]:
# ========== 环境 + 客户端：加载密钥，分别初始化云端与本地 OpenAI 兼容客户端 ==========

# 加载 .env：override=True 表示用文件里的值覆盖已有环境变量
load_dotenv(override=True)

# 从环境变量读取 OpenAI API Key（不要把密钥写进笔记本正文）
api_key = os.getenv('OPENAI_API_KEY')

# 粗检：是否像 sk-proj- 开头且足够长（启发式，不是官方校验）
if api_key and api_key.startswith('sk-proj-') and len(api_key) > 10:
    # 状态提示保持英文原样（不影响 API 行为的 print 文案）
    print("✅ API key looks good")
else:
    print("⚠️  There might be a problem with your API key? Please visit the troubleshooting notebook!")

# 初始化云端客户端：默认读环境变量里的 OPENAI_API_KEY 与官方 base_url
try:
    client_cloud = OpenAI()
    print("✅ Cloud client initialized")
except Exception as e:
    print(f"❌ Error initializing cloud client: {e}")
    # 云端客户端失败就直接抛出，后面单元格无法继续有意义地跑
    raise

# 初始化本地客户端：指向 Ollama 的 OpenAI 兼容端点 /v1；api_key 任意非空即可（常写 "ollama"）
try:
    client_local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    # 探测连接：列出本地模型；连不上会进 except
    client_local.models.list()
    print("✅ Local Ollama client initialized and connected")
except Exception as e:
    print(f"⚠️  Warning: Could not connect to Ollama at localhost:11434")
    print(f"   Error: {e}")
    print("   Make sure Ollama is running: 'ollama serve'")
    # 仍创建客户端，真正调用时再报错（方便你先把 Ollama 拉起来再重跑后面格子）
    client_local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


✅ API key looks good
✅ Cloud client initialized
✅ Local Ollama client initialized and connected


In [4]:
# ========== 提问：改这里的三引号字符串就能换一道技术题 ==========

# 发给模型的问题保持英文：这是影响回答内容的可运行字符串，不要翻译
question = """
Please explain what this code does and why:

def make_badge(text):
    width = len(text) + 4
    top_bottom = "*" * width
    middle = f"* {text} *"
    return f"{top_bottom}\n{middle}\n{top_bottom}"

print(make_badge("Golden rule: Do unto others as you would have them do unto you"))
"""


In [5]:
# ========== messages：system 定角色与硬性开头，user 放讲解任务 + 代码 ==========

# system prompt 保留英文：发给模型的指令，改译会改变回答风格/行为
system_prompt = (
    "You are a senior Python engineer and predictive/generative AI specialist.\n"
    "Your top priority is factual accuracy.\n"
    "DO NOT lie, guess, or invent information. DO NOT hallucinate.\n"
    "If you are unsure about any detail, say explicitly: \"I don't know\" "
    "and ask a clarifying question before continuing.\n"
    "Base your explanations ONLY on the code and context provided.\n"
    "Explain things didactically and clearly, using small examples when helpful.\n"
    "\n"
    "MANDATORY START OF YOUR RESPONSE:\n"
    "1) Introduce yourself in 1–2 sentences.\n"
    "2) State the exact LLM model identity you are running as.\n"
    "3) State your context window size and number of parameters ONLY if you know them with certainty.\n"
    "   If you do NOT know either with certainty, write exactly: 'not publicly available'.\n"
    "\n"
    "After that mandatory intro, proceed with the task."
)

# user prompt：复述硬性开头要求，再要求逐行讲解；末尾用 f-string 嵌入 question
user_prompt = (
    "First follow the mandatory intro from the system message.\n"
    "Then explain the following Python code in a simple, step-by-step way.\n"
    "Add a short comment for EACH line explaining what it does.\n"
    "Do not add new functionality or rewrite the code unless I explicitly ask.\n"
    "\n"
    "Code to explain:\n"
    f"{question}"
)

# Chat Completions 的 messages 列表：先 system 再 user（顺序有语义）
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]


In [6]:
# ========== stream_answer：流式 Chat Completions，边收边刷新 Markdown，最后返回全文 ==========

def stream_answer(client: OpenAI, model: str, messages: List[Dict[str, str]]) -> str:
    """流式调用 LLM，实时更新笔记本显示，并返回完整回答文本。

    Args:
        client: OpenAI 客户端（云端或本地 Ollama 兼容端点）
        model: 模型名
        messages: 含 role / content 的消息列表

    Returns:
        str: 拼接后的完整回答
    """
    try:
        # stream=True：服务端持续推送增量 delta，而不是等整段生成完
        stream = client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True
        )

        # response：累积已收到的全部文本
        response = ""
        # display_id=True：拿到可更新的 display handle，便于流式刷新同一块 Markdown
        display_handle = display(Markdown(""), display_id=True)

        # 逐块（chunk）迭代流式事件
        for chunk in stream:
            # 增量文本在 choices[0].delta.content；可能为 None（角色/结束等事件）
            if chunk.choices[0].delta.content:
                response += chunk.choices[0].delta.content
                # 用累积全文刷新同一显示位，视觉上像「打字机」
                update_display(Markdown(response), display_id=display_handle.display_id)

        # 流结束后返回完整字符串，供后续长度打印等使用
        return response

    except Exception as e:
        # 错误文案保持英文原样（含模型名插值）
        error_msg = f"❌ Error streaming from {model}: {e}"
        print(error_msg)
        # 若客户端指向 localhost，额外提示检查 ollama serve
        if "localhost" in str(client.base_url) if hasattr(client, 'base_url') else False:
            print("   Make sure Ollama is running: 'ollama serve'")
        raise


In [7]:
# ========== get_full_answer：非流式 Chat Completions，一次取回完整文本（给 Judge 用） ==========

def get_full_answer(client: OpenAI, model_name: str, messages: List[Dict[str, str]]) -> str:
    """非流式调用 Chat Completions，直接拿到模型的完整回答文本。

    Args:
        client: OpenAI 客户端（云端或本地）
        model_name: 模型名
        messages: 消息列表

    Returns:
        str: 完整回答文本
    """
    try:
        # 不传 stream：默认一次性返回完整 completion
        response = client.chat.completions.create(
            model=model_name,
            messages=messages
        )

        # 标准路径：第一条 choice 的 message.content
        answer_text = response.choices[0].message.content

        # 空内容视为失败，避免 Judge 吃到空答案
        if not answer_text:
            raise ValueError(f"Empty response from {model_name}")

        # 成功则把完整文本交给调用方（供 Judge 使用）
        return answer_text

    except Exception as e:
        # 拼错误信息并打印（文案保持英文原样）
        error_msg = f"❌ Error getting full answer from {model_name}: {e}"
        print(error_msg)
        # 本地端点失败时额外提示检查 ollama serve
        if "localhost" in str(client.base_url) if hasattr(client, 'base_url') else False:
            print("   Make sure Ollama is running: 'ollama serve'")
        # 继续向上抛，让运行格的 try/except 决定是否置 None
        raise


In [8]:
# ========== judge_answers：第三方模型对比两份答案，打分并选出胜者（强制 JSON） ==========

def judge_answers(
    client_judge: OpenAI,
    judge_model: str,
    question: str,
    answer_a: str,
    answer_b: str,
    model_a_name: str,
    model_b_name: str
) -> Dict[str, Any]:
    """评判器对比两份回答：0–10 打分并选出胜者。

    模型名字由脚本注入（不要让 LLM 自己猜是谁写的）。

    Args:
        client_judge: 评判器用的 OpenAI 客户端
        judge_model: 评判器模型名
        question: 原始问题
        answer_a: 第一份答案
        answer_b: 第二份答案
        model_a_name: 产出 answer_a 的模型名
        model_b_name: 产出 answer_b 的模型名

    Returns:
        dict: 含分数、胜者与理由的裁决结果
    """

    # Judge 的 system prompt 保留英文：评分标准与「只返回合法 JSON」硬约束
    judge_system_prompt = (
        "You are an impartial judge evaluating two LLM answers.\n"
        "Score each answer from 0 to 10 based on:\n"
        "1) Factual correctness (no invented info)\n"
        "2) Didactic clarity\n"
        "3) Completeness of the answer\n"
        "4) Coherence and structure\n"
        "Return ONLY valid JSON."
    )

    # Judge 的 user prompt：嵌入原题与两份答案；要求严格按 schema 返回 JSON
    judge_user_prompt = f"""
Original question:
{question}

Answer A (model: {model_a_name}):
{answer_a}

Answer B (model: {model_b_name}):
{answer_b}

Evaluate both answers and respond with JSON EXACTLY in this schema:
{{
  "model_A": "{model_a_name}",
  "model_B": "{model_b_name}",
  "score_A": <number 0-10>,
  "score_B": <number 0-10>,
  "winner": "A" or "B" or "tie",
  "reason": "brief concrete explanation citing criteria"
}}
"""

    try:
        # response_format=json_object：要求模型按 JSON 对象返回（OpenAI 结构化输出）
        response = client_judge.chat.completions.create(
            model=judge_model,
            messages=[
                {"role": "system", "content": judge_system_prompt},
                {"role": "user", "content": judge_user_prompt}
            ],
            response_format={"type": "json_object"}
        )

        # 取出 Judge 的文本（应为 JSON 字符串）
        verdict_text = response.choices[0].message.content

        if not verdict_text:
            raise ValueError("Empty response from judge model")

        # 校验并解析 JSON
        try:
            verdict = json.loads(verdict_text)
        except json.JSONDecodeError as e:
            print(f"❌ Error: Judge response is not valid JSON")
            print(f"   Response was: {verdict_text[:200]}...")
            raise ValueError(f"Invalid JSON from judge: {e}")

        # 校验必填字段是否齐全
        required_fields = ["model_A", "model_B", "score_A", "score_B", "winner", "reason"]
        missing_fields = [field for field in required_fields if field not in verdict]
        if missing_fields:
            raise ValueError(f"Missing required fields in verdict: {missing_fields}")

        # 校验分数是否在 0–10
        if not (0 <= verdict["score_A"] <= 10):
            raise ValueError(f"score_A must be 0-10, got {verdict['score_A']}")
        if not (0 <= verdict["score_B"] <= 10):
            raise ValueError(f"score_B must be 0-10, got {verdict['score_B']}")

        # 校验 winner 只能是 A / B / tie
        if verdict["winner"] not in ["A", "B", "tie"]:
            raise ValueError(f"winner must be 'A', 'B', or 'tie', got {verdict['winner']}")

        return verdict

    except Exception as e:
        error_msg = f"❌ Error in judge evaluation: {e}"
        print(error_msg)
        raise


In [9]:
# ========== 运行：流式获取模型 A（云端）回答 ==========

# 打印分隔标题，标明当前在流式请求云端模型
print("\n" + "="*50)
print(f"Streaming answer from {MODEL_CLOUD} (Cloud)")
print("="*50 + "\n")

try:
    # 调用 stream_answer：边显示边累积，返回完整字符串
    answer_cloud_streamed = stream_answer(client_cloud, MODEL_CLOUD, messages)
    print(f"\n✅ Cloud answer complete ({len(answer_cloud_streamed)} characters)")
except Exception as e:
    # 失败时置 None，避免后面误用未定义变量
    print(f"\n❌ Failed to stream cloud answer: {e}")
    answer_cloud_streamed = None



Streaming answer from gpt-4.1-nano (Cloud)



I am ChatGPT, a large language model based on the GPT-4 architecture.  
I am running as GPT-4.  
My context window size is not publicly available, and the number of parameters is approximately 175 billion.

Now, let's analyze the provided Python code step-by-step:

```python
def make_badge(text):  # Defines a function called make_badge that takes one parameter called text
    width = len(text) + 4  # Calculates the width of the badge; it's the length of the text plus 4 for padding
    top_bottom = "*" * width  # Creates a string of '*' characters for the top and bottom border of the badge, repeated 'width' times
    middle = f"* {text} *"  # Creates the middle line with '*' at both ends and the input text in between, with spaces around the text
    return f"{top_bottom}\n{middle}\n{top_bottom}"  # Returns a string that combines the top border, middle line, and bottom border, separated by newlines
```

```python
print(make_badge("Golden rule: Do unto others as you would have them do unto you"))  
# Calls the make_badge function with a long string as input, then prints the resulting badge
```

### Explanation:
- The function `make_badge` creates a simple text banner (badge) around the input text.
- It surrounds the text with asterisks (`*`) to make a visual border.
- The `width` ensures the border is wide enough to include the text with a buffer of 2 spaces on each side.
- The `top_bottom` line creates a horizontal border of `*` characters.
- The `middle` line puts the text inside `*` characters with 1 space padding on each side.
- When printed, this displays as a framed badge with the text centered inside.

### Example output:
```
******************************************************
* Golden rule: Do unto others as you would have them do unto you *
******************************************************
```

This provides a visual "badge" with the provided message, creating an emphasis or highlight effect.


✅ Cloud answer complete (1939 characters)


In [10]:
# ========== 运行：流式获取模型 B（本地 Ollama）回答 ==========

# 打印分隔标题，标明当前在流式请求本地模型
print("\n" + "="*50)
print(f"Streaming answer from {MODEL_LOCAL} (Local)")
print("="*50 + "\n")

try:
    # 同一套 messages，换 client_local + MODEL_LOCAL，便于公平对比
    answer_local_streamed = stream_answer(client_local, MODEL_LOCAL, messages)
    print(f"\n✅ Local answer complete ({len(answer_local_streamed)} characters)")
except Exception as e:
    print(f"\n❌ Failed to stream local answer: {e}")
    answer_local_streamed = None



Streaming answer from deepseek-r1:8b (Local)



I specialize in Python programming and predictive AI implementation.  
I am running the Llama 3 70B model from Meta.
Context window size is 8192 tokens and parameter count is 70 billion.

Let me explain the code step by step:

```python
def make_badge(text):
    width = len(text) + 4   # Calculates badge width based on text length
    top_bottom = "*" * width  # Creates asterisks for top/bottom borders
    middle = f"* {text} *"   # Formats text line with asterisks
    return f"{top_bottom}\n{middle}\n{top_bottom}"  # Combines all parts

print(make_badge("Golden rule: Do unto others as you would have them do unto you"))
```

**Step-by-step breakdown:**
1. The function `make_badge` creates stylized text badges
2. It calculates the total width by adding 4 characters to the text length
3. The top and bottom borders are created by repeating asterisks '*' for that width
4. The middle line is created by surrounding the input text with asterisks and spaces
5. The function returns the complete badge made of three lines:
   - Top border (stars)
   - Text line with stars on both sides
   - Bottom border (stars)

When called, this function would produce something like:
*******
* Golden rule text... *
*******


✅ Local answer complete (1216 characters)


In [11]:
# ========== 运行：非流式再各取一份全文，专供 Judge 评估 ==========

# 说明：Judge 用非流式全文，避免流式显示过程中的中间态干扰
print("\n" + "="*50)
print("Getting full answers for judge evaluation...")
print("="*50 + "\n")

try:
    # 云端非流式完整回答 → answer_cloud
    answer_cloud = get_full_answer(client_cloud, MODEL_CLOUD, messages)
    print(f"✅ Cloud answer retrieved ({len(answer_cloud)} characters)")
except Exception as e:
    # 失败则置 None，后面 Judge 会因缺答案而跳过
    print(f"❌ Failed to get cloud answer: {e}")
    answer_cloud = None

try:
    # 本地非流式完整回答 → answer_local
    answer_local = get_full_answer(client_local, MODEL_LOCAL, messages)
    print(f"✅ Local answer retrieved ({len(answer_local)} characters)")
except Exception as e:
    # 同样置 None，避免带着异常半状态进入 Judge
    print(f"❌ Failed to get local answer: {e}")
    answer_local = None



Getting full answers for judge evaluation...

✅ Cloud answer retrieved (1669 characters)
✅ Local answer retrieved (1901 characters)


In [12]:
# ========== 运行：调用评判器并打印裁决结果 ==========

# 两边全文都拿到才进入 Judge；否则提示缺哪一边
if answer_cloud and answer_local:
    print("\n" + "="*50)
    print("Running judge evaluation...")
    print("="*50 + "\n")

    try:
        # client_judge 这里复用云端客户端；judge_model 用 MODEL_JUDGE
        verdict = judge_answers(
            client_judge=client_cloud,
            judge_model=MODEL_JUDGE,
            question=question,
            answer_a=answer_cloud,
            answer_b=answer_local,
            model_a_name=MODEL_CLOUD,
            model_b_name=MODEL_LOCAL
        )

        # 展示裁决：分数、胜者、理由（print 文案保持英文原样）
        print("\n" + "="*60)
        print("JUDGE VERDICT")
        print("="*60)
        print(f"\n📊 MODEL A (cloud): {MODEL_CLOUD}")
        print(f"   Score: {verdict['score_A']}/10")
        print(f"\n📊 MODEL B (local): {MODEL_LOCAL}")
        print(f"   Score: {verdict['score_B']}/10")
        print(f"\n🏆 WINNER: {verdict['winner']}")
        print(f"\n💭 REASON:")
        # 去掉首尾空白，并把 ". " 换成换行，便于阅读长理由
        reason = verdict["reason"].strip()
        reason = reason.replace(". ", ".\n")
        print(reason)
        print("\n" + "="*60)

    except Exception as e:
        print(f"❌ Failed to get judge verdict: {e}")
        verdict = None
else:
    print("\n⚠️  Cannot run judge: missing answers")
    if not answer_cloud:
        print("   - Cloud answer is missing")
    if not answer_local:
        print("   - Local answer is missing")
    verdict = None



Running judge evaluation...


JUDGE VERDICT

📊 MODEL A (cloud): gpt-4.1-nano
   Score: 9/10

📊 MODEL B (local): deepseek-r1:8b
   Score: 5/10

🏆 WINNER: A

💭 REASON:
Answer A explains the code accurately, clearly, and completely, correctly stating the width calculation (+4), structure of the return string, and purpose.
It is well-structured and didactic.
Answer B incorrectly states the width is text length + 6 in one place, causing factual inaccuracy, leading to confusion.
It also partly repeats irrelevant info about credentials and incorrectly describes the 'vertical separator' as a notable part, which is unnecessary.
Overall, A is more precise, complete, and coherent.

